In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Данные соревнования доступны только для чтения в /kaggle/input
print(os.listdir('/kaggle/input'))

In [ ]:
DATA = '/kaggle/input/flight-delays-fall-2018'
# Читаем обучающую и тестовую выборку
train = pd.read_csv('/kaggle/input/competitions/flight-delays-fall-2018/flight_delays_train.csv.zip')
test = pd.read_csv('/kaggle/input/competitions/flight-delays-fall-2018/flight_delays_test.csv.zip')
# Выводим первые 5 строчек таблицы
print('train:', train.shape, '| test:', test.shape)
train.head()

Данные успешно загружены и разделены на обучающую и тестовую выборки. Обучающий набор содержит как числовые, так и категориальные признаки, а также бинарную целевую переменную dep_delayed_15min, в то время как на тестовом наборе нам предстоит спрогнозировать факт задержки рейса. Разметка целевого класса представлена строковыми значениями «Y» и «N», что потребует их предварительного перевода в числовой формат (1 и 0) перед обучением моделей.

In [ ]:
# 1. Проверяем общую размерность таблицы (количество строк и столбцов)
print("Размер обучающей выборки:", train.shape)

# 2. Оцениваем распределение целевой переменной (баланс классов 'N' и 'Y')
print("\nРаспределение ответов:")
print(train['dep_delayed_15min'].value_counts())

# 3. Выводим техническую сводку: типы данных, количество заполненных строк и расход памяти
print("\nИнформация о столбцах:")
train.info()

Набор данных состоит из 100 000 строк и 9 признаков, при этом пропущенные значения (NaN) полностью отсутствуют. В данных наблюдается заметный дисбаланс классов: рейсы без задержки (N) составляют примерно 81% выборки, тогда как задержанные рейсы (Y) — около 19%. Полученный дисбаланс критически важно учесть при выборе стратегии валидации (StratifiedKFold) и метрики оценки качества, чтобы модель не уходила в банальное предсказание мажоритарного класса.

In [ ]:
# Рассчитываем количество рейсов для каждого статуса задержки
counts = train['dep_delayed_15min'].value_counts()

# Строим столбчатую диаграмму с выбранной цветовой схемой
plt.bar(counts.index, counts.values, color=['lightgreen', 'purple'])

# Добавляем информативные подписи к осям и заголовок графика
plt.title('Распределение задержанных и вовремя вылетевших рейсов')
plt.xlabel('Статус рейса (N — вовремя, Y — задержка)')
plt.ylabel('Количество самолетов')

# Добавляем легкую сетку по оси Y для удобства чтения масштаба
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Отображаем график
plt.show()

Столбчатая диаграмма наглядно подтверждает сильный перекос в данных. Количество вовремя отправленных рейсов (N) многократно превышает количество задержанных (Y). Такое распределение требует особого внимания при построении прогнозных моделей, так как стандартные алгоритмы без дополнительной настройки будут склонны игнорировать миноритарный класс задержек.

In [ ]:
# Извлекаем час вылета для обучающей и тестовой выборок (целочисленное деление на 100)
train['Hour'] = train['DepTime'] // 100
test['Hour'] = test['DepTime'] // 100

# Извлекаем минуты вылета для обеих выборок (остаток от деления на 100)
train['Minute'] = train['DepTime'] % 100
test['Minute'] = test['DepTime'] % 100

# Удаляем исходный столбец DepTime из обоих датасетов за ненадобностью
train = train.drop('DepTime', axis=1)
test = test.drop('DepTime', axis=1)

# Проверяем успешность преобразований, выводя первые 5 строк новой таблицы train_df
train.head()

Признак DepTime был успешно декомпозирован на составляющие Hour и Minute. Теперь данные о времени представлены в чистом числовом формате, а структура таблицы полностью готова к дальнейшему кодированию категориальных фич. Выделение часа вылета позволит алгоритмам выявлять такие зависимости, как повышенная вероятность задержек в вечерние часы пик.

In [ ]:
# Очищаем столбцы от префикса "c-" и переводим в целые числа для обучающей выборки
for col in ['Month', 'DayofMonth', 'DayOfWeek']:
    train[col] = train[col].str.replace('c-', '', regex=False).astype(int)

# Делаем то же самое для тестовой выборки, чтобы структура данных полностью совпадала
for col in ['Month', 'DayofMonth', 'DayOfWeek']:
    test[col] = test[col].str.replace('c-', '', regex=False).astype(int)

# Переводим строковый ответ (Y/N) в бинарные числа 1 и 0 методом map
# 1 — рейс задержан, 0 — рейс улетел вовремя
train['dep_delayed_15min'] = train['dep_delayed_15min'].map({'Y': 1, 'N': 0})

# Проверяем финальную структуру обработанной таблицы
train.head()

Все календарные признаки успешно избавлены от строковых префиксов и приведены к типу int. Целевой признак задержки рейса закодирован как 1 и 0. Теперь данные стали полностью числовыми и готовы к этапу кодирования текстовых признаков аэропортов/авиакомпаний (например, через One-Hot Encoding) и последующему обучению базовых моделей.

In [ ]:
# Принудительно кодируем целевую переменную (Y -> 1, N -> 0), если она еще строковая
if train['dep_delayed_15min'].dtype == object:
    train['dep_delayed_15min'] = train['dep_delayed_15min'].map({'Y': 1, 'N': 0})

# Очищаем признаки дат от префиксов "c-" и приводим к int
for col in ['Month', 'DayofMonth', 'DayOfWeek']:
    train[col] = train[col].astype(str).str.replace('c-', '', regex=False).astype(int)
    test[col] = test[col].astype(str).str.replace('c-', '', regex=False).astype(int)

# Выделяем признаки (X) и таргет (y)
X = train.drop('dep_delayed_15min', axis=1)
y = train['dep_delayed_15min'].astype(int)

# Указываем текстовые колонки для CatBoost
cat_features = ['UniqueCarrier', 'Origin', 'Dest']

# Делим выборку на обучение и валидацию (80/20)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Инициализируем и обучаем модель
model = CatBoostClassifier(iterations=300, random_state=42, verbose=50)
model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val))

Выборка была успешно разделена на обучающий и валидационный сплиты, а модель CatBoost обучена за 300 итераций. Лог обучения показывает стабильное снижение ошибки (Logloss) на проверочных данных, которая в финале зафиксировалась на отметке ~0.4150. Это говорит о том, что алгоритм успешно улавливает зависимости в данных и не переобучается, создавая сильную базовую линию (baseline) для дальнейших экспериментов.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# Выбираем только числовые столбцы для простой модели
numeric_cols = ['Month', 'DayofMonth', 'DayOfWeek', 'Distance', 'Hour', 'Minute']

# Создаем саму модель регрессии (задаем настройки)
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Запускаем обучение модели только на числовых столбцах
lr_model.fit(X_train[numeric_cols], y_train)

# Предсказываем вероятности задержки для проверочных рейсов
lr_preds = lr_model.predict_proba(X_val[numeric_cols])[:, 1]

# Считаем размер ошибки модели, сравнивая прогноз с реальными ответами
lr_error = log_loss(y_val, lr_preds)

# LogLoss предыдущей CatBoost
cb_error = 0.4150

print("Результаты сравнения алгоритмов:")
print(f"-> Логистическая регрессия (Logloss): {lr_error:.4f}")
print(f"-> Градиентный бустинг CatBoost (Logloss): {cb_error:.4f}\n")

# Вывод победителя
if cb_error < lr_error:
    print(f"Вывод: Выиграла модель CatBoostClassifier, так как её ошибка {cb_error:.4f} меньше.")
else:
    print(f"Вывод: Выиграла модель Логистическая регрессия, так как её ошибка {lr_error:.4f} меньше.")

Мы дополнительно обучили вторую модель — Логистическую регрессию. Так как она не умеет напрямую работать с текстом, мы обучили её только на числовых признаках. Программа автоматически сравнила результаты по метрике ошибки Logloss. Победил CatBoostClassifier, так как его ошибка (0.4150) заметно меньше, чем у регрессии. Это связано с тем, что CatBoost учитывает влияние конкретных аэропортов и авиакомпаний на задержки рейсов. Для итогового прогноза выбираем именно CatBoost.

In [ ]:
# Загружаем пример файла отправки, чтобы скопировать правильную структуру ID рейсов
submission = pd.read_csv('/kaggle/input/competitions/flight-delays-fall-2018/sample_submission.csv.zip')

# Генерируем предсказания вероятностей класса 1 (задержка рейса) с помощью CatBoost
# Используем метод predict_proba, так как метрикой соревнования является ROC AUC
test_preds = model.predict_proba(test)[:, 1]

# Записываем полученные вероятности в целевую колонку нашего файла отправки
submission['dep_delayed_15min'] = test_preds

# Сохраняем финальный результат в рабочий каталог Kaggle без сохранения индексов строк
submission.to_csv('submission.csv', index=False)

# Выводим первые 5 строк и размер итоговой таблицы для визуального контроля
print("Размер файла отправки:", submission.shape)
submission.head()

Итоговое заключение по практике

В ходе выполнения практики я выполнил задачу и настроил систему предсказания задержек авиарейсов.
Что было сделано:

    1. Подготовка данных: Я почистил даты от лишнего текста, а сплошное время вылета (например, 1934) разделил на обычные часы и минуты. Для того, чтобы модель понимала время суток — т. к. рейсы чаще задерживаются в часы пик, чем ночью.

    2. Проверка моделей: Я протестировал два разных алгоритма: простую Логистическую регрессию и бустинг CatBoost от Яндекса. Чтобы проверка была честной, я отложил 20% данных в сторону и тестировал алгоритмы на рейсах, которые они раньше не видели.

    3. Результат сравнения: Модель CatBoost выиграла с большим отрывом. Её ошибка (Logloss) составила всего 0.4150. Это произошло потому, что CatBoost умеет работать с текстом и сам учел, какие авиакомпании и аэропорты задерживают вылеты чаще других. Регрессия так не умела и работала только с числами.

Итог: Используя лучшую модель (CatBoost), я сделал прогноз для всех новых рейсов и сохранил эти вероятности в файл my_submission.csv. Формат файла полностью совпадает с шаблоном, который требовался в задании.

## Карточка решения

- **Источник данных:** соревнование Kaggle «Flight Delays Fall 2018» (правила и условия использования данных).
- **Ссылка на ноутбук (решение/код): https://www.kaggle.com/code/nailkhairullin/mmz-nail-khairullin
- **Ограничения данных:**
  1) выборка ограничена и может быть нерепрезентативной для всех рейсов;
  2) приватность/лицензия — данные используются только в рамках правил соревнования;
  3) исторический срез (2018) может не отражать текущие закономерности.